# 🧠 第7周-Day2：记忆系统与语义搜索

> Agent 记忆三层结构：Working → Episodic → Semantic
> 今天用 numpy 模拟 embedding + 余弦相似度检索，对比不同记忆层的容量与召回效果。

**实验目标：**
1. 模拟三层记忆的存储与容量差异
2. 用 TF 模拟 embedding，实现余弦相似度检索
3. 可视化检索召回率对比

In [ ]:
# 配置 matplotlib 中文显示
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

## 实验1：记忆三层结构模拟

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple
import time

@dataclass
class MemoryEntry:
    text: str
    embedding: np.ndarray = field(default_factory=lambda: np.random.rand(32))
    timestamp: float = field(default_factory=time.time)
    access_count: int = 0

class WorkingMemory:
    """工作记忆：容量小，速度快，会话内有效"""
    def __init__(self, capacity=5):
        self.capacity = capacity
        self.items: List[MemoryEntry] = []

    def add(self, entry: MemoryEntry):
        if len(self.items) >= self.capacity:
            self.items.pop(0)  # FIFO 淘汰
        self.items.append(entry)

class EpisodicMemory:
    """情景记忆：按时间存储，容量较大"""
    def __init__(self, capacity=50):
        self.capacity = capacity
        self.items: List[MemoryEntry] = []

    def add(self, entry: MemoryEntry):
        if len(self.items) >= self.capacity:
            self.items.pop(0)
        self.items.append(entry)

class SemanticMemory:
    """语义记忆：按概念去重，长期有效"""
    def __init__(self, capacity=200):
        self.capacity = capacity
        self.items: List[MemoryEntry] = []
        self._threshold = 0.85  # 相似度阈值：超过则合并

    def add(self, entry: MemoryEntry):
        # 简化：如果已有相似概念则跳过
        for existing in self.items:
            sim = np.dot(entry.embedding, existing.embedding) / (
                np.linalg.norm(entry.embedding) * np.linalg.norm(existing.embedding) + 1e-8)
            if sim > self._threshold:
                existing.access_count += 1
                return  # 去重合并
        if len(self.items) >= self.capacity:
            self.items.pop(0)
        self.items.append(entry)

# 创建记忆系统
working = WorkingMemory()
episodic = EpisodicMemory()
semantic = SemanticMemory()

for i in range(10):
    e = MemoryEntry(text=f"对话片段{i}", embedding=np.random.rand(32))
    working.add(e); episodic.add(e); semantic.add(e)

print(f"Working Memory: {len(working.items)}/5")
print(f"Episodic Memory: {len(episodic.items)}/50")
print(f"Semantic Memory: {len(semantic.items)}/200")

## 实验2：模拟语义检索（余弦相似度）

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# 构建知识库：10条记忆
knowledge = [
    "用户偏好暗色主题", "上周购买了Python书籍", "下午3点有会议",
    "项目截止日期是周五", "用户英语水平较高", "上月提交了Bug报告",
    "偏好简洁的回复格式", "常用工具是VSCode", "上次讨论了API设计",
    "用户角色是技术负责人"
]
kb_embeddings = np.random.RandomState(42).rand(len(knowledge), 32)

# 用户查询
query = "用户喜欢什么样的界面风格？"
query_emb = kb_embeddings[0] + np.random.normal(0, 0.1, 32)  # 模拟相似查询

scores = [cosine_sim(query_emb, kb_embeddings[i]) for i in range(len(knowledge))]
ranked = sorted(zip(knowledge, scores), key=lambda x: -x[1])

print("检索结果：")
for i, (text, score) in enumerate(ranked[:5]):
    bar = "█" * int(score * 20)
    print(f"  {i+1}. [{score:.3f}] {bar} {text}")

## 实验3：记忆层容量与检索对比可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：容量对比
layers = ["Working", "Episodic", "Semantic"]
caps = [5, 50, 200]
ax = axes[0]
ax.bar(layers, caps, color=["#FF6B6B", "#4ECDC4", "#45B7D1"])
ax.set_ylabel("容量（条目）")
ax.set_title("三层记忆容量（对数尺度）")
ax.set_yscale("log")

# 右图：模拟 Top-K 召回率
k_values = [1, 3, 5, 10]
recall_working = [0.8, 0.6, 0.4, 0.2]
recall_episodic = [0.7, 0.75, 0.8, 0.85]
recall_semantic = [0.6, 0.7, 0.85, 0.95]

ax = axes[1]
ax.plot(k_values, recall_working, 'o-', label="Working", color="#FF6B6B")
ax.plot(k_values, recall_episodic, 's-', label="Episodic", color="#4ECDC4")
ax.plot(k_values, recall_semantic, '^-', label="Semantic", color="#45B7D1")
ax.set_xlabel("Top-K")
ax.set_ylabel("召回率")
ax.set_title("不同记忆层 Top-K 召回率（模拟）")
ax.legend()
plt.tight_layout()
plt.show()